# Ticker Price Targets vs Price From Financial Modeling Prep

This notebook pulls the latest consensus price target snapshot for any ticker from FMP's `price-target-consensus` endpoint, then reconstructs a historical target band from FMP's analyst target history so we can compare price target trends against the stock's price.

Change `ticker_str` in `../../params.py` and rerun the notebook. Inputs like `BRK\\B`, `BRK/B`, and `BRK.B` are normalized automatically for FMP.

Note: FMP's stable `price-target-consensus` endpoint returns the current snapshot only. The historical high, low, median, and consensus series below are reconstructed from FMP's `price-target-news` history using a trailing 365-day window of analyst targets.

In [ ]:
from pathlib import Path
import os
import runpy

import pandas as pd
import plotly.graph_objects as go
import requests


base_path_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PARAMS_FILE_CANDIDATES = []
for base_path in base_path_candidates:
    PARAMS_FILE_CANDIDATES.extend(
        [
            base_path / "params.py",
            base_path / "Single Asset Profile" / "params.py",
            base_path / "Notebooks" / "Single Asset Profile" / "params.py",
            base_path / "Investment Scripts" / "Notebooks" / "Single Asset Profile" / "params.py",
        ]
    )
for params_file in PARAMS_FILE_CANDIDATES:
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()

TICKER = single_asset_params["ticker_str"]  # Shared Single Asset Profile ticker.
ROLLING_WINDOW_DAYS = 365

In [ ]:
BASE_URL = "https://financialmodelingprep.com/stable"


def find_project_root(start_path: Path | None = None) -> Path:
    current = (start_path or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    return current


def load_project_env() -> None:
    env_path = find_project_root() / ".env"
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in {'"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)


def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def normalize_symbol(ticker: str) -> str:
    normalized = str(ticker).strip().upper()
    for old, new in ((chr(92), "."), ("/", "."), ("-", ".")):
        normalized = normalized.replace(old, new)
    return normalized


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload


def fetch_all_price_target_news(symbol: str, page_size: int = 100, max_pages: int = 100) -> pd.DataFrame:
    pages = []

    for page in range(max_pages):
        payload = request_json("price-target-news", symbol=symbol, limit=page_size, page=page)
        page_frame = pd.DataFrame(payload)
        if page_frame.empty:
            break
        pages.append(page_frame)
        if len(page_frame) < page_size:
            break

    if not pages:
        raise RuntimeError(f"FMP did not return any price-target news rows for {symbol}.")

    target_news = pd.concat(pages, ignore_index=True)
    target_news = target_news.drop_duplicates().copy()

    if "publishedDate" not in target_news.columns:
        raise RuntimeError(f"FMP price-target-news response for {symbol} is missing publishedDate.")

    target_news["targetDate"] = pd.to_datetime(
        target_news["publishedDate"], errors="coerce", utc=True
    ).dt.tz_convert(None).dt.normalize()
    target_news["priceTarget"] = pd.to_numeric(target_news.get("priceTarget"), errors="coerce")
    target_news["adjPriceTarget"] = pd.to_numeric(
        target_news.get("adjPriceTarget", target_news["priceTarget"]),
        errors="coerce",
    )
    target_news["priceWhenPosted"] = pd.to_numeric(target_news.get("priceWhenPosted"), errors="coerce")
    target_news["effectiveTarget"] = target_news["adjPriceTarget"].where(
        target_news["adjPriceTarget"].notna(),
        target_news["priceTarget"],
    )

    target_news = target_news.dropna(subset=["targetDate", "effectiveTarget"]).sort_values(
        ["targetDate", "effectiveTarget", "newsTitle"],
        kind="mergesort",
    ).reset_index(drop=True)

    if target_news.empty:
        raise RuntimeError(f"FMP returned price-target news for {symbol}, but none contained usable targets.")

    return target_news


def build_historical_target_band(
    price_history: pd.DataFrame,
    target_news: pd.DataFrame,
    rolling_window_days: int,
) -> pd.DataFrame:
    if price_history.empty:
        return pd.DataFrame(
            columns=["date", "targetLow", "targetMedian", "targetConsensus", "targetHigh", "targetCount"]
        )

    history = price_history.loc[:, ["date"]].copy()
    history["date"] = pd.to_datetime(history["date"], errors="coerce")
    history = history.dropna(subset=["date"]).drop_duplicates(subset=["date"]).sort_values("date")

    targets = target_news.loc[:, ["targetDate", "effectiveTarget"]].copy()
    targets["targetDate"] = pd.to_datetime(targets["targetDate"], errors="coerce")
    targets["effectiveTarget"] = pd.to_numeric(targets["effectiveTarget"], errors="coerce")
    targets = targets.dropna(subset=["targetDate", "effectiveTarget"]).sort_values("targetDate")

    band_rows = []
    rolling_window = pd.Timedelta(days=rolling_window_days)

    for current_date in history["date"]:
        window_start = current_date - rolling_window
        active_targets = targets.loc[
            (targets["targetDate"] >= window_start) & (targets["targetDate"] <= current_date),
            "effectiveTarget",
        ]
        if active_targets.empty:
            continue

        band_rows.append(
            {
                "date": current_date,
                "targetLow": active_targets.min(),
                "targetMedian": active_targets.median(),
                "targetConsensus": active_targets.mean(),
                "targetHigh": active_targets.max(),
                "targetCount": int(active_targets.count()),
            }
        )

    return pd.DataFrame(band_rows)


SYMBOL = normalize_symbol(TICKER)
SYMBOL

In [ ]:
quote_snapshot = request_json("quote", symbol=SYMBOL)
company_name = quote_snapshot[0].get("name", SYMBOL) if quote_snapshot else SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL

consensus_snapshot = pd.DataFrame(request_json("price-target-consensus", symbol=SYMBOL))
target_news = fetch_all_price_target_news(SYMBOL)
price_history = pd.DataFrame(request_json("historical-price-eod/light", symbol=SYMBOL))

price_history["date"] = pd.to_datetime(price_history["date"])
price_history["price"] = pd.to_numeric(price_history["price"], errors="coerce")
price_history = (
    price_history.loc[price_history["date"] >= target_news["targetDate"].min(), ["date", "price"]]
    .sort_values("date")
    .reset_index(drop=True)
)

consensus_snapshot


In [ ]:
target_band = build_historical_target_band(price_history, target_news, ROLLING_WINDOW_DAYS)
plot_df = price_history.merge(target_band, on="date", how="left")

latest_reconstructed = plot_df.dropna(subset=["targetConsensus"]).iloc[-1]
latest_snapshot = consensus_snapshot.iloc[0]

comparison = pd.DataFrame(
    [
        {
            "series": f"Trailing {ROLLING_WINDOW_DAYS}-day reconstructed band",
            "date": latest_reconstructed["date"].date(),
            "targetLow": round(latest_reconstructed["targetLow"], 2),
            "targetMedian": round(latest_reconstructed["targetMedian"], 2),
            "targetConsensus": round(latest_reconstructed["targetConsensus"], 2),
            "targetHigh": round(latest_reconstructed["targetHigh"], 2),
            "targetCount": int(latest_reconstructed["targetCount"]),
        },
        {
            "series": "Latest FMP price-target-consensus snapshot",
            "date": pd.Timestamp.today().date(),
            "targetLow": latest_snapshot["targetLow"],
            "targetMedian": latest_snapshot["targetMedian"],
            "targetConsensus": latest_snapshot["targetConsensus"],
            "targetHigh": latest_snapshot["targetHigh"],
            "targetCount": None,
        },
    ]
)

comparison


In [ ]:
latest_market_price = (
    float(quote_snapshot[0].get("price"))
    if quote_snapshot and quote_snapshot[0].get("price") is not None
    else float(plot_df["price"].dropna().iloc[-1])
)

series_targets = [
    {
        "series": f"Trailing {ROLLING_WINDOW_DAYS}-day reconstructed band",
        "referenceDate": latest_reconstructed["date"].date(),
        "referencePrice": float(latest_reconstructed["price"]),
        "targetLow": latest_reconstructed["targetLow"],
        "targetMedian": latest_reconstructed["targetMedian"],
        "targetConsensus": latest_reconstructed["targetConsensus"],
        "targetHigh": latest_reconstructed["targetHigh"],
    },
    {
        "series": "Latest FMP price-target-consensus snapshot",
        "referenceDate": pd.Timestamp.today().date(),
        "referencePrice": latest_market_price,
        "targetLow": latest_snapshot["targetLow"],
        "targetMedian": latest_snapshot["targetMedian"],
        "targetConsensus": latest_snapshot["targetConsensus"],
        "targetHigh": latest_snapshot["targetHigh"],
    },
]

target_label_map = {
    "targetLow": "Low target",
    "targetMedian": "Median target",
    "targetConsensus": "Consensus target",
    "targetHigh": "High target",
}

target_position_rows = []
for target_set in series_targets:
    reference_price = target_set["referencePrice"]

    for key, label in target_label_map.items():
        target_value = pd.to_numeric(target_set[key], errors="coerce")
        if pd.isna(target_value) or target_value == 0 or reference_price == 0:
            continue

        pct_above_below_target = ((reference_price / target_value) - 1) * 100
        pct_move_to_target = ((target_value / reference_price) - 1) * 100

        target_position_rows.append(
            {
                "series": target_set["series"],
                "referenceDate": target_set["referenceDate"],
                "referencePrice": round(reference_price, 2),
                "targetLine": label,
                "targetValue": round(float(target_value), 2),
                "position": "Above" if pct_above_below_target > 0 else "Below" if pct_above_below_target < 0 else "At",
                "pctAboveBelowTarget": round(pct_above_below_target, 2),
                "pctMoveToTarget": round(pct_move_to_target, 2),
            }
        )

target_position_summary = pd.DataFrame(target_position_rows)
target_position_summary

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["price"],
        name=f"{SYMBOL} close",
        line={"color": "#e5eefc", "width": 2.5},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetConsensus"],
        name=f"Consensus target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#60a5fa", "width": 2.5},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetMedian"],
        name=f"Median target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#fbbf24", "width": 2.5},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetHigh"],
        name=f"High target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#4ade80", "dash": "dash"},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetLow"],
        name=f"Low target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#f87171", "dash": "dash"},
        fill="tonexty",
        fillcolor="rgba(96, 165, 250, 0.14)",
    )
)

fig.add_trace(
    go.Scatter(
        x=[plot_df["date"].iloc[-1]],
        y=[latest_snapshot["targetConsensus"]],
        mode="markers+text",
        name="Latest consensus API snapshot",
        marker={"color": "#38bdf8", "size": 11, "symbol": "diamond"},
        text=[f"API snapshot: {latest_snapshot['targetConsensus']:.2f}"],
        textfont={"color": "#dbeafe"},
        textposition="top center",
    )
)

fig.update_layout(
    title=f"{chart_label} price vs analyst price target band",
    template="plotly_dark",
    paper_bgcolor="#020817",
    plot_bgcolor="#0f172a",
    font={"color": "#e2e8f0"},
    hovermode="x unified",
    hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    xaxis_title="Date",
    yaxis_title="USD per share",
    autosize=True,
    height=680,
    margin={"l": 60, "r": 30, "t": 90, "b": 60},
)

fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True)
fig.update_yaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True)

fig.show(config={"responsive": True, "displaylogo": False})


In [ ]:
from plotly.subplots import make_subplots

historical_target_distance = plot_df.loc[
    :,
    ["date", "price", "targetLow", "targetMedian", "targetConsensus", "targetHigh"],
] .copy()

target_distance_columns = {
    "targetLow": "Low target",
    "targetMedian": "Median target",
    "targetConsensus": "Consensus target",
    "targetHigh": "High target",
}

for column_name, label in target_distance_columns.items():
    target_values = pd.to_numeric(historical_target_distance[column_name], errors="coerce").replace(0, pd.NA)
    historical_target_distance[label] = (
        (historical_target_distance["price"] / target_values) - 1
    ) * 100

relative_colors = {
    "Low target": "#f87171",
    "Median target": "#fbbf24",
    "Consensus target": "#60a5fa",
    "High target": "#4ade80",
}
positive_fill_color = "rgba(34, 197, 94, 0.22)"
negative_fill_color = "rgba(248, 113, 113, 0.22)"

relative_fig = make_subplots(
    rows=len(relative_colors),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=list(relative_colors.keys()),
)

for row_number, (label, color) in enumerate(relative_colors.items(), start=1):
    trace_df = historical_target_distance.loc[:, ["date", label]].dropna().copy()
    trace_df["zero"] = 0.0
    trace_df["positive"] = trace_df[label].where(trace_df[label] >= 0)
    trace_df["negative"] = trace_df[label].where(trace_df[label] < 0)

    relative_fig.add_trace(
        go.Scatter(
            x=trace_df["date"],
            y=trace_df["zero"],
            mode="lines",
            line={"color": "rgba(0, 0, 0, 0)", "width": 0},
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row_number,
        col=1,
    )
    relative_fig.add_trace(
        go.Scatter(
            x=trace_df["date"],
            y=trace_df["positive"],
            mode="lines",
            line={"color": "rgba(0, 0, 0, 0)", "width": 0},
            fill="tonexty",
            fillcolor=positive_fill_color,
            hoverinfo="skip",
            showlegend=False,
            connectgaps=False,
        ),
        row=row_number,
        col=1,
    )
    relative_fig.add_trace(
        go.Scatter(
            x=trace_df["date"],
            y=trace_df["zero"],
            mode="lines",
            line={"color": "rgba(0, 0, 0, 0)", "width": 0},
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row_number,
        col=1,
    )
    relative_fig.add_trace(
        go.Scatter(
            x=trace_df["date"],
            y=trace_df["negative"],
            mode="lines",
            line={"color": "rgba(0, 0, 0, 0)", "width": 0},
            fill="tonexty",
            fillcolor=negative_fill_color,
            hoverinfo="skip",
            showlegend=False,
            connectgaps=False,
        ),
        row=row_number,
        col=1,
    )
    relative_fig.add_trace(
        go.Scatter(
            x=trace_df["date"],
            y=trace_df[label],
            name=label,
            line={"color": color, "width": 2.5},
            hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2f}%<extra>" + label + "</extra>",
            showlegend=False,
        ),
        row=row_number,
        col=1,
    )
    relative_fig.add_hline(
        y=0,
        line_dash="dot",
        line_color="#cbd5e1",
        opacity=0.8,
        row=row_number,
        col=1,
    )
    relative_fig.update_yaxes(
        title_text=label,
        ticksuffix="%",
        showgrid=True,
        gridcolor="rgba(148, 163, 184, 0.18)",
        zeroline=False,
        automargin=True,
        row=row_number,
        col=1,
    )

relative_fig.update_layout(
    title=(
        f"{chart_label} price premium/discount to analyst target lines"
        "<br><sup>Green shading = price above target. Red shading = price below target.</sup>"
    ),
    template="plotly_dark",
    paper_bgcolor="#020817",
    plot_bgcolor="#0f172a",
    font={"color": "#e2e8f0"},
    hovermode="x unified",
    hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
    autosize=True,
    height=1100,
    margin={"l": 90, "r": 30, "t": 110, "b": 60},
)

relative_fig.update_xaxes(
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=False,
    automargin=True,
)
relative_fig.update_xaxes(title_text="Date", row=len(relative_colors), col=1)

relative_fig.show(config={"responsive": True, "displaylogo": False})